In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
BATCH_SIZE = 100_000
SEED = 37
SAMPLE_PROPORTION = 0.0005

In [3]:
member_batches = []
max_index = 0

for batch in pd.read_csv('../data/members_v3.csv', chunksize = BATCH_SIZE):
    member_batches.append(batch.sample(frac = SAMPLE_PROPORTION, random_state = SEED))
    if max(batch.index) + 1 > max_index:
        max_index = max(batch.index) + 1

sample_members = pd.concat(member_batches, ignore_index = True)
member_ids = set(sample_members['msno'])

print(f'{len(sample_members): ,} out of {max_index: ,} members selected.')
print()
print(sample_members.head())
print()
print(sample_members.info())

 3,385 out of  6,769,473 members selected.

                                           msno  city  bd  gender  \
0  59LT0TXKCMT0x8lBV6lihO8CvTZ9z5Sb9UsypwfC6Uk=     1   0     NaN   
1  +9WO11eKjgcyXUN4ewrT4Q00zTKY/fHbZgYaxZsSDt8=     1   0     NaN   
2  6P/ta5JtN8UDFiuHI2ELWXLaRUsgN97Zj63QAMWLycQ=     1   0     NaN   
3  0g6qJ7am8vecp54eFV7I0ceLy8yzHWJA0a32aNZM6YU=     1  21  female   
4  LVZhPn7ds6KtnZ1GPMHV4DFhCuYAiQtf4QucKYL2d9A=     1   0     NaN   

   registered_via  registration_init_time  
0               4                20161202  
1               9                20120208  
2               4                20161206  
3               3                20150414  
4               9                20150919  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3385 entries, 0 to 3384
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   msno                    3385 non-null   object
 1   city     

In [4]:
sample_members_typecast = sample_members.copy()

sample_members_typecast[['city', 'registered_via']] = sample_members_typecast[['city', 'registered_via']].astype('object')
sample_members_typecast['registration_init_time'] = pd.to_datetime(sample_members_typecast['registration_init_time'], format = '%Y%m%d')

print(sample_members_typecast.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3385 entries, 0 to 3384
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   msno                    3385 non-null   object        
 1   city                    3385 non-null   object        
 2   bd                      3385 non-null   int64         
 3   gender                  1151 non-null   object        
 4   registered_via          3385 non-null   object        
 5   registration_init_time  3385 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 158.8+ KB
None


In [5]:
transaction_batches = []

max_index_total = 0
for csv in ['../data/transactions.csv', '../data/transactions_v2.csv']:
    max_index = 0
    for batch in pd.read_csv(csv, chunksize = BATCH_SIZE):
        transaction_batches.append(batch[batch['msno'].isin(member_ids)])
        if max(batch.index) + 1 > max_index:
            max_index = max(batch.index) + 1
    max_index_total += max_index
    
sample_transactions = pd.concat(transaction_batches, ignore_index = True)

print(f'{len(sample_transactions): ,} out of {max_index_total: ,} transactions selected.')
print()
print(sample_transactions.head())
print()
print(sample_transactions.info())

 10,056 out of  22,978,755 transactions selected.

                                           msno  payment_method_id  \
0  wcXslluldERi48gljH+9o9KgXgYsCt5ih8+i3EXUNIE=                 34   
1  mCQIOwBybuLG6L88zC65GiNd5jihPflNq03KWRYbA+c=                 40   
2  qaT4OYJ+gEbriiBlR29k4BUJeszVX8VxapTeUuO0xa4=                 40   
3  pW3ywgOqWZkR2vrSJKz+j6T8NuC0Msi6Q6XZdZMMDwE=                 39   
4  nBkJ6ORyG8iptHWFrP4/4e4AVkIWnVCklerE8JpQQK8=                 39   

   payment_plan_days  plan_list_price  actual_amount_paid  is_auto_renew  \
0                 30              149                 149              1   
1                 30              149                 149              1   
2                 30              149                 149              1   
3                 30              149                 149              1   
4                 30              149                 149              1   

   transaction_date  membership_expire_date  is_cancel  
0          201

In [6]:
sample_transactions_typecast = sample_transactions.copy()

sample_transactions_typecast['payment_method_id'] = sample_transactions_typecast['payment_method_id'].astype('object')
for column in ['transaction_date', 'membership_expire_date']:
    sample_transactions_typecast[column] = pd.to_datetime(sample_transactions_typecast[column], format = '%Y%m%d')
sample_transactions_typecast[['is_auto_renew', 'is_cancel']] = sample_transactions_typecast[['is_auto_renew', 'is_cancel']].astype('boolean')

print(sample_transactions_typecast.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10056 entries, 0 to 10055
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   msno                    10056 non-null  object        
 1   payment_method_id       10056 non-null  object        
 2   payment_plan_days       10056 non-null  int64         
 3   plan_list_price         10056 non-null  int64         
 4   actual_amount_paid      10056 non-null  int64         
 5   is_auto_renew           10056 non-null  boolean       
 6   transaction_date        10056 non-null  datetime64[ns]
 7   membership_expire_date  10056 non-null  datetime64[ns]
 8   is_cancel               10056 non-null  boolean       
dtypes: boolean(2), datetime64[ns](2), int64(3), object(2)
memory usage: 589.3+ KB
None


In [14]:
sample = pd.merge(sample_members_typecast, sample_transactions_typecast, how = 'inner').drop_duplicates()
sample['bd'] = [None if x <= 0 or x > 111 else x for x in sample['bd']]
print(sample.info())

<class 'pandas.core.frame.DataFrame'>
Index: 10054 entries, 0 to 10055
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   msno                    10054 non-null  object        
 1   city                    10054 non-null  object        
 2   bd                      5267 non-null   float64       
 3   gender                  5257 non-null   object        
 4   registered_via          10054 non-null  object        
 5   registration_init_time  10054 non-null  datetime64[ns]
 6   payment_method_id       10054 non-null  object        
 7   payment_plan_days       10054 non-null  int64         
 8   plan_list_price         10054 non-null  int64         
 9   actual_amount_paid      10054 non-null  int64         
 10  is_auto_renew           10054 non-null  boolean       
 11  transaction_date        10054 non-null  datetime64[ns]
 12  membership_expire_date  10054 non-null  datetime64[

In [15]:
print(f'proportion of missing ages: {sample['bd'].isna().mean():.2f}')
print(f'proportion of missing genders: {sample['gender'] .isna().mean():.2f}')
conditional_prop = sample[sample['bd'].isna()]['gender'].isna().mean()
print(f'proportion of missing genders given that age is missing: {conditional_prop:.2f}')
conditional_prop_reverse = sample[sample['gender'].isna()]['bd'].isna().mean()
print(f'proportion of missing ages given that gender is missing: {conditional_prop_reverse:.2f}')
incomplete_prop = ((sample['bd'].isna()) | (sample['gender'].isna())).mean()
print(f'proportion of incomplete profiles: {incomplete_prop:.2f}')

proportion of missing ages: 0.48
proportion of missing genders: 0.48
proportion of missing genders given that age is missing: 0.98
proportion of missing ages given that gender is missing: 0.98
proportion of incomplete profiles: 0.49


In [16]:
wrong_expire = ((sample['membership_expire_date'] - sample['transaction_date']).dt.days != sample['payment_plan_days']).mean()
print(f'proportion of mismatched expirations: {wrong_expire:.2f}')

proportion of mismatched expirations: 0.71


In [17]:
sample2 = sample.copy()

sample2['membership_expire_date'] = sample2['transaction_date'] + sample2['payment_plan_days'].apply(lambda x: pd.Timedelta(days = x))
sample2 = sample2.drop_duplicates().reset_index(drop = True)

wrong_expire2 = ((sample2['membership_expire_date'] - sample2['transaction_date']).dt.days != sample2['payment_plan_days']).mean()
print(f'proportion of mismatched updated expirations: {wrong_expire2:.2f}')

proportion of mismatched updated expirations: 0.00


In [18]:
print('payment plan feature correlations:')
print(sample2[['payment_plan_days', 'plan_list_price', 'actual_amount_paid']].corr())

payment plan feature correlations:
                    payment_plan_days  plan_list_price  actual_amount_paid
payment_plan_days            1.000000         0.985272            0.962132
plan_list_price              0.985272         1.000000            0.975018
actual_amount_paid           0.962132         0.975018            1.000000


In [19]:
sample3 = sample2.copy().drop(columns = ['payment_plan_days', 'plan_list_price']).drop_duplicates().reset_index(drop = True)

In [20]:
print(
    sample3.copy()
    .assign(proportion = lambda x: 1/len(x))
    .groupby(['is_auto_renew', 'is_cancel'])
    .agg({'proportion': 'sum'})
    .reset_index()
)

   is_auto_renew  is_cancel  proportion
0          False      False    0.150515
1           True      False    0.816982
2           True       True    0.032503
